*0.2 Math / ML basics*

# sampling: temperature

**The situation.** Two features share one model. The invoice extractor must give the same answer every time. The marketing assistant must not write the same slogan twice. Same model, opposite needs — and one setting handles both.

**Temperature.** Before softmax, the model's scores are divided by the temperature. Below 1 the gaps grow, the top token takes almost everything, and answers become repeatable. Above 1 the gaps shrink, unlikely tokens get a real chance, and answers vary. Temperature 0 means "always take the top token".

In [1]:
# Load OPENAI_API_KEY from the .env file. The OpenAI clients read it from the environment.
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())
MODEL = "gpt-4o-mini"

**The math, on real probabilities.** Take the five candidates from the previous item as scores and re-softmax them at three temperatures.

In [2]:
import torch
from openai import OpenAI

client = OpenAI(timeout=30)
response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Name one colour."}],
    max_tokens=1,
    logprobs=True,
    top_logprobs=5,
)
candidates = response.choices[0].logprobs.content[0].top_logprobs
tokens = []
logprobs = []
for candidate in candidates:
    tokens.append(candidate.token)
    logprobs.append(candidate.logprob)
scores = torch.tensor(logprobs)

temperatures = (0.2, 1.0, 2.0)
header = f"{'token':<10}"
rows = {}
for temperature in temperatures:
    header += f"{'T=' + str(temperature):>10}"
    rows[temperature] = torch.softmax(scores / temperature, dim=0)
print(header)
for index, token in enumerate(tokens):
    line = f"{token!r:<10}"
    for temperature in temperatures:
        line += f"{rows[temperature][index]:>10.1%}"
    print(line)
assert rows[0.2][0] > rows[1.0][0] > rows[2.0][0]

token          T=0.2     T=1.0     T=2.0
'Blue'        100.0%     98.9%     83.5%
'Azure'         0.0%      0.5%      6.0%
'Te'            0.0%      0.3%      4.7%
'Tur'           0.0%      0.1%      3.2%
'Cer'           0.0%      0.1%      2.5%


**Reading the output.** At T=0.2 the top colour takes nearly everything. At T=2.0 the shares even out and the fourth or fifth colour becomes a realistic pick. Same model, same scores — only the division changed.

**Now for real.** Ask five times at temperature 0 and five times at 1.8, count the distinct answers.

In [3]:
answers = {}
for temperature in (0.0, 1.8):
    seen = set()
    for _ in range(5):
        reply = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": "Name one colour. One word."}],
            max_tokens=3,
            temperature=temperature,
        )
        seen.add(reply.choices[0].message.content.strip().lower().strip("."))
    answers[temperature] = seen
    print(f"temperature {temperature}: {len(seen)} distinct answer(s) → {sorted(seen)}")
assert len(answers[0.0]) <= len(answers[1.8])

temperature 0.0: 1 distinct answer(s) → ['blue']


temperature 1.8: 2 distinct answer(s) → ['blue', 'scarlet']


**The rule to remember.** Temperature 0 for anything a program reads (extraction, classification, code). Around 0.7–1.0 for text a person reads. Above 1.2 only when variety is the point, and expect some nonsense.

| Use it when | Don't when | Instead use |
|---|---|---|
| tuning the repeatability/variety trade-off | you need *guaranteed* identical output — T=0 is very repeatable, not perfectly | caching the answer |

**Watch out**
- Temperature 0 is not deterministic across hardware or model updates. Cache if you must have the same bytes.
- Temperature and top-p both make output more or less varied; change one at a time or you cannot tell what did what.
- High temperature also raises the chance of broken JSON, invented facts and switched languages.